<a href="https://colab.research.google.com/github/ouzkalem/DLSG/blob/main/W2_exercise_SOLN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# inzvaDLSG Regularization and Hyperparameter Tuning Notebook Solution

---
## Section 1: Imports & Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.metrics import accuracy_score
import random
from itertools import product

# Set random seeds for reproducibility
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

print("All imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

---
## Section 2: Data Loading - SOLUTIONS

In [ ]:
transform = transforms.ToTensor()

# SOLUTION TODO 1: Load Fashion MNIST datasets
train_dataset = torchvision.datasets.FashionMNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.FashionMNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Image shape: {train_dataset[0][0].shape}")

In [ ]:
# SOLUTION TODO 2: Split training data
train_size = int(0.9 * len(train_dataset))  # 54,000
val_size = len(train_dataset) - train_size   # 6,000
train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])

print(f"Training split: {len(train_dataset)} samples")
print(f"Validation split: {len(val_dataset)} samples")

In [ ]:
# SOLUTION TODO 3: Create DataLoaders
BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

sample_batch, sample_labels = next(iter(train_loader))
print(f"Batch shape: {sample_batch.shape}")
print(f"Labels shape: {sample_labels.shape}")

---
## Section 3: Model Architecture - SOLUTIONS

In [ ]:
class FashionMNIST_NN(nn.Module):
    """
    SOLUTION TODOs 4 & 5: Complete model implementation
    """

    def __init__(self, dropout_rate=0.5):
        super(FashionMNIST_NN, self).__init__()

        # SOLUTION TODO 4: Define layers
        self.fc1 = nn.Linear(784, 256)   # Input: 28*28 = 784, Hidden: 256
        self.fc2 = nn.Linear(256, 128)   # Hidden: 256 → 128
        self.fc3 = nn.Linear(128, 10)    # Hidden: 128 → Output: 10 classes
        self.dropout = nn.Dropout(dropout_rate)
        self.relu = nn.ReLU()

    def forward(self, x):
        # SOLUTION TODO 5: Forward pass
        # Step 1: Flatten (batch, 1, 28, 28) → (batch, 784)
        x = x.view(x.size(0), -1)

        # Step 2: Pass through layers
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)

        x = self.fc2(x)
        x = self.relu(x)
        x = self.dropout(x)

        out = self.fc3(x)  # No activation - CrossEntropyLoss expects logits

        return out

In [ ]:
# Verify model
test_model_check = FashionMNIST_NN(dropout_rate=0.5)
test_input = torch.randn(64, 1, 28, 28)
test_output = test_model_check(test_input)
print(f"Model output shape: {test_output.shape} (expected: torch.Size([64, 10]))")
print(f"Total parameters: {sum(p.numel() for p in test_model_check.parameters()):,}")

---
## Section 4: Weight Initialization

In [ ]:
def initialize_weights(model, method='kaiming'):
    for layer in model.children():
        if isinstance(layer, nn.Linear):
            if method == 'xavier':
                nn.init.xavier_uniform_(layer.weight)
            elif method == 'kaiming':
                nn.init.kaiming_uniform_(layer.weight, nonlinearity='relu')
            if layer.bias is not None:
                nn.init.zeros_(layer.bias)
    print(f"Weights initialized using {method} method")

---
## Section 5: Early Stopping

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, delta=0):
        self.patience = patience
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.delta = delta
        self.best_val_loss = float('inf')

    def __call__(self, val_loss, model):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        self.best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model_checkpoint.pth')
        print(f"Checkpoint saved! (val_loss: {val_loss:.4f})")

---
## Section 6: Training Function - SOLUTIONS

In [ ]:
def train_model(l1_lambda=0, l2_lambda=0.0, dropout_rate=0.1, weight_init='kaiming', lr=0.01):
    """
    SOLUTION TODOs 6-10: Complete training implementation
    """

    # SOLUTION TODO 6: Create model
    model = FashionMNIST_NN(dropout_rate=dropout_rate)
    initialize_weights(model, weight_init)

    # SOLUTION TODO 7: Create optimizer with L2 regularization (weight_decay)
    optimizer = optim.SGD(model.parameters(), lr=lr, weight_decay=l2_lambda)

    # SOLUTION TODO 8: Create loss function
    criterion = nn.CrossEntropyLoss()

    early_stopping = EarlyStopping(patience=5)

    num_epochs = 100
    train_losses = []
    val_losses = []

    print("=" * 60)
    print("TRAINING STARTED")
    print("=" * 60)

    for epoch in range(num_epochs):
        # TRAINING PHASE
        model.train()
        train_loss = 0.0

        for batch_idx, (inputs, labels) in enumerate(train_loader):
            # SOLUTION TODO 9: Training step

            # Step A: Forward pass
            outputs = model(inputs)

            # Step B: Compute loss
            loss = criterion(outputs, labels)

            # Step C: Add L1 regularization
            if l1_lambda > 0:
                l1_norm = sum(p.abs().sum() for p in model.parameters())
                loss = loss + l1_lambda * l1_norm

            # Step D: Zero gradients
            optimizer.zero_grad()

            # Step E: Backward pass
            loss.backward()

            # Step F: Update weights
            optimizer.step()

            train_loss += loss.item()

        # VALIDATION PHASE
        model.eval()
        val_loss = 0.0

        with torch.no_grad():
            for inputs, labels in val_loader:
                # SOLUTION TODO 10: Validation step
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                if l1_lambda > 0:
                    l1_norm = sum(p.abs().sum() for p in model.parameters())
                    loss = loss + l1_lambda * l1_norm
                val_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss   = val_loss   / len(val_loader)

        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)

        print(f"Epoch [{epoch+1:2d}/{num_epochs}] | "
              f"Train Loss: {avg_train_loss:.4f} | "
              f"Val Loss: {avg_val_loss:.4f}")

        early_stopping(avg_val_loss, model)
        if early_stopping.early_stop:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

    print("=" * 60)
    print("TRAINING COMPLETED")
    print(f"Best validation loss: {early_stopping.best_val_loss:.4f}")
    print("=" * 60)

    model.load_state_dict(torch.load('best_model_checkpoint.pth'))

    return model, train_losses, val_losses

---
## Section 7: Testing Function - SOLUTION

In [ ]:
def test_model(model):
    """
    SOLUTION TODO 11: Complete test implementation
    """
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            #SOLUTION TODO 11: Test prediction
            outputs = model(inputs)
            preds = outputs.argmax(dim=1)  # Get class with highest score

            all_preds.append(preds)
            all_labels.append(labels)

    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)

    test_acc = accuracy_score(all_labels.numpy(), all_preds.numpy())

    print(f"TEST ACCURACY: {test_acc:.4f} ({test_acc*100:.2f}%)")

    return test_acc

---
## Section 8: Visualization - SOLUTION

In [ ]:
def plot_losses(train_losses, val_losses):
    """
    SOLUTION TODO 12: Complete plotting implementation
    """
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Training Loss', marker='o', markersize=4)
    plt.plot(val_losses, label='Validation Loss', marker='s', markersize=4)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss Over Time')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

---
## Section 9: Run the Experiment

# Ablation Study Instructions

Here you can run your ablation studies manually. What we recommend is that you change the hyperparameters below and also the number of weights in the model. You might also adjust the early stopping patience parameter. You can have a few objectives:

- Get ≈90%-95% test accuracy score
- Try to overfit and observe the validation and training curves
- Try to underfit and observe it
- Observe how different hyperparameters actually affect the training (for example, you can set the learning rate to 100, set lambda_1 to 1, or set dropout to 0.9 and observe what kind of different scenarios you encounter)

In order to do these objectives, you should understand the code and change the correct parameters. If you are able to apply these different settings and accomplish these scenarios, that is a good sign.


In [ ]:
# Default hyperparameters
l1_lambda = 0.5
l2_lambda = 0.0
dropout_rate = 0.0
weight_init = 'kaiming'
learning_rate = 0.01

print("\n" + "=" * 60)
print("HYPERPARAMETER CONFIGURATION")
print("=" * 60)
print(f"   L1 Lambda:      {l1_lambda}")
print(f"   L2 Lambda:      {l2_lambda}")
print(f"   Dropout Rate:   {dropout_rate}")
print(f"   Weight Init:    {weight_init}")
print(f"   Learning Rate:  {learning_rate}")
print("=" * 60 + "\n")

In [ ]:
# Train the model
model, train_losses, val_losses = train_model(
    l1_lambda=l1_lambda,
    l2_lambda=l2_lambda,
    dropout_rate=dropout_rate,
    weight_init=weight_init,
    lr=learning_rate
)

In [ ]:
# Plot training curves
plot_losses(train_losses, val_losses)

In [ ]:
# Evaluate on test set
test_acc = test_model(model)

---
## Section 10: Grid Search - SOLUTION

In [ ]:
param_grid = {
    'lr': [0.001, 0.01],
    'dropout_rate': [0.3, 0.5],
    'l1_lambda': [0.0, 0.001],
    'l2_lambda': [0.0, 0.001],
    'weight_init': ['xavier', 'kaiming']
}

In [ ]:
def grid_search(param_grid):
    """
    SOLUTION TODOs 13 & 14: Complete grid search implementation
    """
    best_val_loss = float('inf')
    best_params = None
    best_model = None

    # SOLUTION TODO 13: Generate combinations
    keys = list(param_grid.keys())
    combinations = list(product(*param_grid.values()))

    print(f"Grid Search: Testing {len(combinations)} combinations\n")

    for i, combo in enumerate(combinations):
        params = dict(zip(keys, combo))

        print(f"\n[{i+1}/{len(combinations)}] Testing: {params}")

        # SOLUTION TODO 14: Train and track best
        model, train_losses, val_losses = train_model(**params)
        final_val_loss = val_losses[-1]

        if final_val_loss < best_val_loss:
            best_val_loss = final_val_loss
            best_params = params
            best_model = model
            print(f"New best, Val Loss: {best_val_loss:.4f}")

    print(f"\n{'=' * 60}")
    print(f"GRID SEARCH COMPLETE")
    print(f"Best Validation Loss: {best_val_loss:.4f}")
    print(f"Best Parameters: {best_params}")
    print(f"{'=' * 60}")

    return best_model, best_params

---
## Section 11: Random Search - SOLUTION

In [ ]:
def random_search(param_grid, n_iterations=10):
    """
    SOLUTION TODO 15: Complete random search implementation
    """
    best_val_loss = float('inf')
    best_params = None
    best_model = None

    print(f"Random Search: Testing {n_iterations} random combinations\n")

    for i in range(n_iterations):
        # Sample random values for each hyperparameter
        params = {key: random.choice(values) for key, values in param_grid.items()}

        print(f"\n[{i+1}/{n_iterations}] Testing: {params}")

        # Train and evaluate
        model, train_losses, val_losses = train_model(**params)
        final_val_loss = val_losses[-1]

        if final_val_loss < best_val_loss:
            best_val_loss = final_val_loss
            best_params = params
            best_model = model
            print(f"New best, Val Loss: {best_val_loss:.4f}")

    print(f"\n{'=' * 60}")
    print(f"RANDOM SEARCH COMPLETE")
    print(f"Best Validation Loss: {best_val_loss:.4f}")
    print(f"Best Parameters: {best_params}")
    print(f"{'=' * 60}")

    return best_model, best_params

In [ ]:
# Uncomment to run hyperparameter search

print("\n" + "=" * 60)
print("RUNNING GRID SEARCH (This might take a while...)")
print("=" * 60)
best_model_grid, best_params_grid = grid_search(param_grid)
test_model(best_model_grid)